# Day 1.3 — Structured Outputs

Clear instructions improved answers but did not create a reliable application contract. Now we build:

```text
Question → model → schema-shaped JSON → Pydantic validation → Python object
```

## Before you begin

### Learning outcomes

Define a Pydantic contract, request structured data, and handle validation failure.

Architecture reference: [D02](../../diagrams/source/day_01.md).

### Expected observation

Valid data becomes a typed object; plausible data outside field constraints is rejected.


## Concept briefing

## Why structured output matters

Free-form text is useful for people but unreliable for software. A program cannot safely
assume every response contains the same headings, fields or value types. A schema turns
this ambiguity into a contract. Validation does not make the model correct; it makes a
particular class of failure visible.

Consider a confidence field. The sentence "confidence is high" may be understandable to
a person but difficult to compare. A schema can require a number between 0 and 1. If the
model returns `4.5`, validation rejects it instead of quietly sending bad data deeper into
the application.

The correct mental model is:

- schema validity asks whether the response has an acceptable shape;
- factual evaluation asks whether its claims are correct;
- policy asks whether a requested action is permitted.

These are different checks and should not be collapsed into one model prompt.


## Learning objectives

Explain why formatted text is not automatically valid data, define a Pydantic contract, request JSON Schema output through OpenRouter, and handle invalid data.

In [ ]:
import json,os
from types import SimpleNamespace
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel,Field,ValidationError
load_dotenv(); api_key=os.getenv("OPENROUTER_API_KEY")
client=OpenAI(base_url="https://openrouter.ai/api/v1",api_key=api_key) if api_key else None
COURSE_MODEL=os.getenv("OPENROUTER_MODEL","openai/gpt-oss-120b")
print("Route:","OpenRouter" if client else "mock fallback")


## Build: ask for JSON using words only

This often appears to work, but fields, types, and extra prose are not guaranteed.

In [ ]:
if client:
    plain=client.chat.completions.create(model=COURSE_MODEL,messages=[{"role":"user","content":"Explain an AI agent. Return JSON with topic, summary, key_points, and confidence."}],max_tokens=500,extra_body={"reasoning":{"effort":"low","exclude":True}})
    plain_text=plain.choices[0].message.content
else:
    plain_text='{"topic":"AI agents","summary":"A model-guided application","key_points":["May request tools"],"confidence":0.8}'
print(plain_text)


## Break it mentally

What if confidence is `high`, key points are one paragraph, Markdown surrounds the JSON, or a required field is missing? These are application-data failures.

## Improve: define the contract with Pydantic

In [ ]:
class ResearchSummary(BaseModel):
    topic: str
    summary: str
    key_points: list[str] = Field(min_length=1, max_length=5)
    confidence: float = Field(ge=0.0, le=1.0)

schema = ResearchSummary.model_json_schema()
schema

## Request schema-constrained output

OpenRouter standardizes structured-output requests for compatible models/providers. The application still validates the returned boundary.

In [ ]:
if client:
    response=client.chat.completions.create(model=COURSE_MODEL,messages=[{"role":"user","content":"Explain an AI agent for a beginner with two or three key points."}],response_format={"type":"json_schema","json_schema":{"name":"research_summary","strict":True,"schema":schema}},max_tokens=600,extra_body={"reasoning":{"effort":"low","exclude":True},"provider":{"require_parameters":True}})
    response_text=response.choices[0].message.content
else:
    response_text=json.dumps({"topic":"AI agents","summary":"An application that uses a model to choose bounded actions.","key_points":["The host executes tools","The loop needs limits"],"confidence":0.9})
result=ResearchSummary.model_validate_json(response_text)
result


In [ ]:
print(result.topic)
print(result.confidence)
for number, point in enumerate(result.key_points, start=1):
    print(f"{number}. {point}")

## Observe validation rejecting plausible but invalid data

In [ ]:
invalid_data = '''{
  "topic": "AI agents",
  "summary": "A short summary",
  "key_points": ["Uses a model"],
  "confidence": 4.5
}'''

try:
    ResearchSummary.model_validate_json(invalid_data)
except ValidationError as error:
    print(error)

## Exercise and checkpoint

Create `EngineeringConcept` with name, plain explanation, one-to-three applications, and difficulty from 1–5. Request and validate one concept. Invalid difficulty and empty applications must fail.

We now have `model output → schema validation → Python object`. The model still cannot obtain outside information or reliably perform calculations; tools solve that next.

## Your turn

Add one constrained field and deliberately supply an invalid value.

## Recap

A schema makes failure visible; it does not make model claims correct.
